This is a test implementation for the ID3 decision tree. If it works well, it will be moved to our implementations.py file. ID3 assumes all features are categorical, so we have to split numerical data into categories to make use of them.

In [1]:
# Imports.
import sys
from implementations import *
import numpy as np
from helpers import *
from typing import List
from math import log2
from ID3 import *

In [2]:
# Input the data

# set hyperparams
k = 3 # bin size
max_depth = 6 # max ID3 tree depth
#

path_to_dataset = "data/dataset"
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(
    path_to_dataset, max_rows=5000, max_features=40, NaNstrat="fill"
) # loading is too slow, I use max_rows and max_features for testing -M

# I'm skipping preprocessing as I don't need to standardize when I'm going to split the numerical data into bins either way.
# Ideally I still remove 0-variance columns.
mask = x_train.std(axis=0) != 0
x_train = x_train[:, mask]

# ID3 specific: convert the labels to strings
y_train = y_train[:].astype(str).reshape(-1, 1)
# Discretize numeric features
tx = x_train
bins = compute_bins(tx, k)
tx_disc = apply_bins(tx, bins)
x_test_disc = apply_bins(x_test, bins)



In [14]:
# Format the test data for the ID3 function
def ID3_format(x, y):
    dummy_header = np.array([f"col{i}" for i in range(x.shape[1])]) # we got rid of feature names in preprocess and I don't want to change that code so I assign some names here

    train_data = np.hstack((x, y)) # slap x and y together to comply to ID3 method's format
    return dummy_header, train_data


In [15]:
# TODO data split for cross validation and confusion matrix
# This is the test implementation for kfold cross validation. If it works, it should be moved to helpers.py.

def kfold_inds(n_samples : int, k : int, seed=42):
    """
    Returns start and end indices of k folds on n_samples samples.
    Any remainder is added to the last fold.
    Returns: a list of pairs (2-element lists) of start and end indices respectively.
    """
    np.random.seed(seed)
    inds = []
    
    for i in range(k):
        inds.append([n_samples//k * i, n_samples//k * (i+1) - 1])
    inds[-1][1] = n_samples - 1 # add the rest to the last fold
    return inds



In [16]:

# Train the model.
def test_hyperparams(x_train, y_train, max_depth: int, k=3, seed = 42):
    """
    max_depth - max depth of ID3 tree
    k - number of folds in k-fold cross validation 
    Returns: the depth of the best model. Ties are broken by returning the simplest model i.e. least depth.
    """
    assert len(x_train) == len(y_train)

    # permute the data randomly
    if seed is not None:
        np.random.seed(seed)
        perm = np.random.permutation(len(x_train))
        x_train = x_train[perm]
        y_train = y_train[perm]
    
    folds = kfold_inds(len(x_train), k)
    best_score = -1.0
    
    for depth in range(1, max_depth + 1):
        print(f"Testing depth = {depth}")
        scores = []
        
        best_score_y_train = []
        best_score_y_test = []

        for fold in folds:
            start, end = fold

            # Split into validation and training
            xi_test = x_train[start:end]
            yi_test = y_train[start:end]

            xi_train = np.concatenate((x_train[:start], x_train[end:]), axis=0)
            yi_train = np.concatenate((y_train[:start], y_train[end:]), axis=0)

            header, train_data = ID3_format(xi_train, yi_train)

            model = ID3()
            model.fit(header, train_data, depth, verbose=False)

            header_test, test_data = ID3_format(xi_test, yi_test)
            predictions = model.predict(header_test, test_data, verbose=False)

            acc = np.mean(predictions == yi_test)
            scores.append(acc)

        mean_acc = np.mean(scores)
        print(f"Mean accuracy (depth={depth}): {mean_acc:.4f}")

        if mean_acc > best_score:
            best_score = mean_acc
            best_depth = depth

    print(f"\nBest depth = {best_depth} with mean accuracy = {best_score:.4f}")
    return best_depth    

In [19]:
# Find out the best ID3 tree depth
best_depth = test_hyperparams(tx_disc, y_train, max_depth)

Testing depth = 1
Mean accuracy (depth=1): 0.9117
Testing depth = 2
Mean accuracy (depth=2): 0.9117
Testing depth = 3
Mean accuracy (depth=3): 0.9117
Testing depth = 4
Mean accuracy (depth=4): 0.9112
Testing depth = 5
Mean accuracy (depth=5): 0.9085
Testing depth = 6
Mean accuracy (depth=6): 0.9053

Best depth = 1 with mean accuracy = 0.9117


In [20]:
# Fit the best model we found
dummy_column = np.full((tx_disc.shape[0], 1), "x") # placeholder because ID3::predict expects a placeholder last column
test_dummy_header, test_data = ID3_format(tx_disc, dummy_column)
final_dummy_header, final_train_data = ID3_format(tx, y_train) 
best_model = ID3()
best_model.fit(final_dummy_header, final_train_data, best_depth)
# Generate predictions on it
predictions = best_model.predict(test_dummy_header, test_data)

In [21]:
print(predictions)

['-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1